# Phase 5 — Model Training

This notebook demonstrates the full training pipeline for both IrisNet variants:

| Model | Head | Loss |
|---|---|---|
| `IrisNet_softmax` | Dense(num_classes, softmax) | CategoricalCrossentropy |
| `IrisNet_ArcFace` | ArcFaceLayer(margin=0.5, scale=64) | CategoricalCrossentropy(from_logits=True) |

### Demo mode (this notebook)
The cells below run **5 epochs** so the notebook can be executed quickly to verify the pipeline end-to-end.  
For the full training run (50 epochs, EarlyStopping) use the command line:

```bash
# From the project root, with the venv activated:
python -m src.models.train_softmax
python -m src.models.train_arcface
```

> **Hardware note:** Full training on the MSI Bravo 15 (AMD Radeon via DirectML) is expected to take several hours per model. Run overnight and load the saved `models/softmax_best.h5` / `models/arcface_best.h5` for evaluation in Phase 6.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
import matplotlib.pyplot as plt
import tensorflow as tf

from src.utils.data_loader import build_datasets
from src.models.train_softmax import build_softmax_model
from src.models.train_arcface import build_arcface_model, _adapt_dataset_for_arcface
from src.models.train_softmax import get_callbacks as softmax_callbacks
from src.models.train_arcface import get_callbacks as arcface_callbacks

DEMO_EPOCHS = 5
BATCH_SIZE  = 32

print('TensorFlow:', tf.__version__)
print('GPUs available:', len(tf.config.list_physical_devices('GPU')))

In [ ]:
# ── Build datasets ────────────────────────────────────────────────────────────
train_ds, val_ds, test_ds, NUM_CLASSES = build_datasets(
    processed_root='../data/processed',
    batch_size=BATCH_SIZE,
)
print(f'Classes: {NUM_CLASSES}')
print(f'Train batches: {len(train_ds)}   Val batches: {len(val_ds)}')

## Part 1 — Softmax Baseline Training

In [ ]:
softmax_model = build_softmax_model(num_classes=NUM_CLASSES)
softmax_model.summary()

In [ ]:
import os
os.makedirs('../models', exist_ok=True)

# Override checkpoint paths so notebook outputs go to models/ relative to project root
import src.models.train_softmax as ts
ts.CHECKPOINT_PATH = '../models/softmax_best.h5'
ts.HISTORY_PATH    = '../models/softmax_history.json'

history_softmax = softmax_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=DEMO_EPOCHS,
    callbacks=softmax_callbacks(),
    verbose=1,
)

In [ ]:
def plot_history(history, title):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    epochs = range(1, len(history.history['loss']) + 1)

    ax1.plot(epochs, history.history['loss'],     'b-o', label='Train loss')
    ax1.plot(epochs, history.history['val_loss'], 'r-o', label='Val loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title(f'{title} — Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(epochs, history.history['accuracy'],     'b-o', label='Train acc')
    ax2.plot(epochs, history.history['val_accuracy'], 'r-o', label='Val acc')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.set_title(f'{title} — Accuracy')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

plot_history(history_softmax, 'IrisNet Softmax (demo: 5 epochs)')

## Part 2 — ArcFace Training

In [ ]:
arcface_model, arcface_backbone = build_arcface_model(num_classes=NUM_CLASSES)
arcface_model.summary()

In [ ]:
import src.models.train_arcface as ta
ta.CHECKPOINT_PATH = '../models/arcface_best.h5'
ta.BACKBONE_PATH   = '../models/arcface_backbone.h5'
ta.HISTORY_PATH    = '../models/arcface_history.json'

train_ds_af = _adapt_dataset_for_arcface(train_ds)
val_ds_af   = _adapt_dataset_for_arcface(val_ds)

history_arcface = arcface_model.fit(
    train_ds_af,
    validation_data=val_ds_af,
    epochs=DEMO_EPOCHS,
    callbacks=arcface_callbacks(),
    verbose=1,
)

In [ ]:
plot_history(history_arcface, 'IrisNet ArcFace (demo: 5 epochs)')

## Summary

Both pipelines ran successfully for the demo duration.  
To train to convergence, run from the project root:

```bash
python -m src.models.train_softmax
python -m src.models.train_arcface
```

Trained model weights will be saved to `models/`. Proceed to Phase 6 once training completes.

In [ ]:
# ── Side-by-side val loss comparison ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
e = range(1, DEMO_EPOCHS + 1)
ax.plot(e, history_softmax.history['val_loss'], 'b-o', label='Softmax val loss')
ax.plot(e, history_arcface.history['val_loss'], 'r-o', label='ArcFace val loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Loss')
ax.set_title('Softmax vs ArcFace — Validation Loss (demo)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()